# How LLMs Work: A Step-by-Step Guide with Code

This notebook walks through the core concepts of how Large Language Models work, with runnable code examples to build intuition.

**Learning objectives:**
- Understand what tokens are and how tokenization works
- See the difference between training and inference
- Explore attention and transformer architecture concepts
- Understand transfer learning and adaptation strategies

---

## Step 0: The Shortest Definition

An LLM is a model that predicts the **next token** given previous tokens.

Everything else (chatting, reasoning, tool use) is built on top of that core capability.

Let's see this in action with a simple example:

In [ ]:
# Conceptual example: next-token prediction
# This is what an LLM fundamentally does

def simple_next_token_demo():
    """Demonstrate the core LLM concept: predicting the next token."""
    
    # Imagine the model has seen: "The cat sat on the"
    context = ["The", "cat", "sat", "on", "the"]
    
    # The model outputs probabilities for what comes next
    next_token_probabilities = {
        "mat": 0.35,
        "floor": 0.25,
        "bed": 0.15,
        "chair": 0.10,
        "roof": 0.05,
        # ... thousands more tokens with tiny probabilities
    }
    
    print(f"Context: {' '.join(context)}")
    print(f"\nNext token probabilities (top 5):")
    for token, prob in sorted(next_token_probabilities.items(), key=lambda x: -x[1]):
        print(f"  '{token}': {prob:.0%}")
    
    # The model samples from this distribution
    # With temperature=0, it always picks the highest probability
    # With higher temperature, it's more random
    
simple_next_token_demo()

---

## Step 1: Tokens (What the Model Actually Sees)

You type text. A **tokenizer** turns it into tokens (integers). The model predicts the next token.

### Why tokenization matters

| Concept | Implication |
|---------|-------------|
| **Subword units** | "unhappiness" might be ["un", "happiness"] or ["un", "happ", "iness"] |
| **Context length** | Models have a max token limit (4K, 8K, 128K, 1M+) |
| **Token ≠ word** | ~1.3 tokens per English word on average; varies by language |
| **Cost** | API pricing is per token, not per word |

Let's explore tokenization with code:

In [ ]:
# First, let's install tiktoken (OpenAI's tokenizer)
# Uncomment the line below if you need to install it
# !pip install tiktoken

In [ ]:
import tiktoken

# Get the tokenizer used by GPT-4 and GPT-3.5-turbo
enc = tiktoken.get_encoding("cl100k_base")

def explore_tokenization(text):
    """Show how text gets tokenized."""
    tokens = enc.encode(text)
    
    print(f"Text: '{text}'")
    print(f"Number of tokens: {len(tokens)}")
    print(f"Token IDs: {tokens}")
    print(f"Tokens decoded individually:")
    for i, token_id in enumerate(tokens):
        token_str = enc.decode([token_id])
        print(f"  [{i}] ID={token_id:6d} -> '{token_str}'")
    print()

# Try different examples
explore_tokenization("Hello, world!")
explore_tokenization("The quick brown fox jumps over the lazy dog.")
explore_tokenization("unhappiness")
explore_tokenization("GPT-4 is a large language model.")

In [ ]:
# Interesting tokenization examples

examples = [
    "hello",           # common word
    "Hello",           # capitalized (different token!)
    " hello",          # leading space (different token!)
    "hello!",          # with punctuation
    "supercalifragilisticexpialidocious",  # long word
    "def fibonacci(n):\n    if n <= 1:\n        return n",  # code
    "こんにちは",        # Japanese
    "🎉🚀",            # emojis
]

print("Token counts for different inputs:\n")
for text in examples:
    tokens = enc.encode(text)
    print(f"{len(tokens):3d} tokens | {repr(text)}")

In [ ]:
# Why does this matter? Cost estimation!

def estimate_cost(text, price_per_1k_tokens=0.01):
    """Estimate API cost for a given text."""
    tokens = enc.encode(text)
    cost = (len(tokens) / 1000) * price_per_1k_tokens
    return len(tokens), cost

sample_prompt = """
You are a helpful assistant. Please analyze the following text and provide:
1. A brief summary (2-3 sentences)
2. Key themes identified
3. Sentiment analysis (positive/negative/neutral)

Text to analyze:
The quick brown fox jumps over the lazy dog. This pangram contains every letter 
of the English alphabet at least once. It has been used since at least the late 
19th century for testing typewriters and computer keyboards.
"""

num_tokens, cost = estimate_cost(sample_prompt)
print(f"Prompt tokens: {num_tokens}")
print(f"Estimated input cost: ${cost:.4f} (at $0.01/1K tokens)")

---

## Step 2: Training vs Inference

### Training (Pretraining)

```
┌─────────────────────────────────────────────────────────────┐
│                     PRETRAINING                             │
│                                                             │
│   Internet text ──► Tokenize ──► Predict next token ──► Loss│
│   (TB of data)                   (self-supervised)          │
│                                                             │
│   Result: Base model with language understanding            │
└─────────────────────────────────────────────────────────────┘
```

### Post-training (Alignment)

```
┌─────────────────────────────────────────────────────────────┐
│                    POST-TRAINING                            │
│                                                             │
│   Supervised Fine-tuning (SFT)                              │
│   ──► Human demonstrations of good responses                │
│                                                             │
│   RLHF / DPO                                                │
│   ──► Human preferences: "Response A is better than B"      │
│                                                             │
│   Result: Model follows instructions, refuses harmful tasks │
└─────────────────────────────────────────────────────────────┘
```

Let's build a tiny language model to understand the training process:

In [ ]:
import random
from collections import defaultdict

class SimpleBigramLM:
    """
    A simple bigram language model.
    Predicts the next token based only on the previous token.
    
    This is NOT how modern LLMs work, but it illustrates the core concept:
    learning to predict the next token from data.
    """
    
    def __init__(self):
        # counts[prev_token][next_token] = count
        self.counts = defaultdict(lambda: defaultdict(int))
        self.totals = defaultdict(int)
    
    def train(self, text):
        """Train on a text corpus by counting bigrams."""
        # Simple tokenization: split on whitespace
        tokens = text.lower().split()
        
        # Count transitions
        for i in range(len(tokens) - 1):
            prev_token = tokens[i]
            next_token = tokens[i + 1]
            self.counts[prev_token][next_token] += 1
            self.totals[prev_token] += 1
    
    def get_next_token_probs(self, prev_token):
        """Get probability distribution over next tokens."""
        prev_token = prev_token.lower()
        if prev_token not in self.counts:
            return {}
        
        total = self.totals[prev_token]
        return {
            token: count / total 
            for token, count in self.counts[prev_token].items()
        }
    
    def generate(self, start_token, max_tokens=20):
        """Generate text starting from a token."""
        tokens = [start_token.lower()]
        
        for _ in range(max_tokens):
            probs = self.get_next_token_probs(tokens[-1])
            if not probs:
                break
            
            # Sample from the distribution
            next_token = random.choices(
                list(probs.keys()), 
                weights=list(probs.values())
            )[0]
            tokens.append(next_token)
        
        return " ".join(tokens)

In [ ]:
# Train our tiny model on some text

training_text = """
The cat sat on the mat. The cat slept on the bed. The dog ran in the park.
The dog played with the ball. The cat chased the mouse. The mouse ran away.
The bird flew over the tree. The tree stood in the garden. The garden was beautiful.
The sun shone brightly. The sky was blue. The clouds drifted slowly.
The cat watched the bird. The dog watched the cat. The mouse hid in the hole.
"""

model = SimpleBigramLM()
model.train(training_text)

# See what the model learned
print("What comes after 'the'?")
probs = model.get_next_token_probs("the")
for token, prob in sorted(probs.items(), key=lambda x: -x[1])[:5]:
    print(f"  '{token}': {prob:.1%}")

In [ ]:
# Generate some text

print("Generated text samples:\n")
for start in ["the", "cat", "dog"]:
    for i in range(3):
        generated = model.generate(start, max_tokens=10)
        print(f"  Starting with '{start}': {generated}")
    print()

### Key insight

Our bigram model only looks at **one previous token**. Real LLMs look at **thousands of tokens** (the entire context window) and use **attention** to decide what's relevant.

**Practical takeaway**: At inference time, the model is not "searching the internet" or "looking things up" unless you give it tools or retrieved documents.

---

## Step 3: Why Transformers Changed Everything

Transformers are the dominant architecture behind modern LLMs. At a high level, a transformer block does two things:

### A) Mix information across tokens (self-attention)

```
Query (Q) ──┐
            ├──► Attention weights ──► Weighted sum of Values
Key (K) ────┘
Value (V) ─────────────────────────────────────────────────►
```

- Each token computes **Query**, **Key**, and **Value** vectors
- Attention score = softmax(Q · K^T / √d)
- Output = weighted sum of Values

### B) Transform that mixed information (feedforward layers)

After mixing, the model applies non-linear transformations (MLP) to compute richer features.

Let's implement a simplified attention mechanism:

In [ ]:
import numpy as np

def softmax(x):
    """Compute softmax values for each row."""
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def simple_attention(query, key, value):
    """
    Simplified attention mechanism.
    
    Args:
        query: (seq_len, d_k) - what we're looking for
        key: (seq_len, d_k) - what each position offers
        value: (seq_len, d_v) - the actual content
    
    Returns:
        output: (seq_len, d_v) - weighted combination of values
        attention_weights: (seq_len, seq_len) - who attends to whom
    """
    d_k = query.shape[-1]
    
    # Compute attention scores: Q @ K^T / sqrt(d_k)
    scores = np.matmul(query, key.T) / np.sqrt(d_k)
    
    # Apply softmax to get attention weights
    attention_weights = softmax(scores)
    
    # Weighted sum of values
    output = np.matmul(attention_weights, value)
    
    return output, attention_weights

In [ ]:
# Demo: attention on a simple sentence

# Imagine we have 4 tokens: ["The", "cat", "sat", "down"]
# Each token has a 3-dimensional embedding

np.random.seed(42)

# Simulated embeddings (in reality, these are learned)
tokens = ["The", "cat", "sat", "down"]
seq_len = len(tokens)
d_model = 4  # embedding dimension

# Random embeddings for demonstration
embeddings = np.random.randn(seq_len, d_model)

# In a real transformer, Q, K, V are linear projections of embeddings
# Here we'll just use the embeddings directly for simplicity
Q = embeddings
K = embeddings
V = embeddings

output, attention_weights = simple_attention(Q, K, V)

print("Attention weights (who attends to whom):")
print("Rows = query token, Columns = key token\n")

# Print header
print(f"{'':>8}", end="")
for t in tokens:
    print(f"{t:>8}", end="")
print()

# Print attention matrix
for i, t in enumerate(tokens):
    print(f"{t:>8}", end="")
    for j in range(seq_len):
        print(f"{attention_weights[i, j]:>8.2f}", end="")
    print()

In [ ]:
# Visualize attention weights

try:
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(attention_weights, cmap='Blues')
    
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens)
    ax.set_yticklabels(tokens)
    
    ax.set_xlabel('Key (attending to)')
    ax.set_ylabel('Query (from)')
    ax.set_title('Attention Weights')
    
    # Add colorbar
    plt.colorbar(im)
    
    # Add text annotations
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            text = ax.text(j, i, f'{attention_weights[i, j]:.2f}',
                          ha='center', va='center', color='black', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("matplotlib not installed. Install with: pip install matplotlib")

### The Full Transformer Picture

```
Input tokens
     │
     ▼
┌─────────────────┐
│ Token Embedding │
│ + Positional    │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Transformer    │ ◄── Repeat N times (e.g., 32, 80, 128 layers)
│     Block       │
│                 │
│ ┌─────────────┐ │
│ │ Self-Attn   │ │
│ └─────────────┘ │
│ ┌─────────────┐ │
│ │ Feedforward │ │
│ └─────────────┘ │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ Output Layer    │
│ (vocab logits)  │
└─────────────────┘
         │
         ▼
   Next token
```

**Intuition**:
- When predicting "it", the model attends to the noun "it" refers to
- When generating code, it attends to variable names and brackets far earlier

---

## Step 4: Transfer Learning (Why You Can Adapt Models)

Most useful LLM behavior comes from transfer learning:

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│  Pretrain    │ ──► │   Adapt      │ ──► │   Deploy     │
│  (general)   │     │ (specific)   │     │  (your app)  │
└──────────────┘     └──────────────┘     └──────────────┘
```

### Adaptation Strategies

| Strategy | When to use | Pros | Cons |
|----------|-------------|------|------|
| **Prompting** | Always start here | No training, fast iteration | Limited by context window |
| **RAG** | Need fresh/private knowledge | No weight changes, updatable | Retrieval quality matters |
| **Fine-tuning** | Need consistent style/format | Persistent behavior change | Needs data, can overfit |
| **RLHF/DPO** | Align to preferences | Better than SFT for preferences | Complex, needs comparisons |

**Practical takeaway**: You rarely train from scratch; you start from a base model and choose an adaptation strategy.

In [ ]:
# Example: Different prompting strategies

def show_prompting_strategies():
    """Demonstrate different prompting approaches."""
    
    task = "Classify the sentiment of: 'This movie was absolutely terrible.'"
    
    strategies = {
        "Zero-shot": f"""
{task}

Sentiment:""",
        
        "Few-shot": f"""
Classify the sentiment of the following texts.

Text: "I loved this book!"
Sentiment: Positive

Text: "The service was okay, nothing special."
Sentiment: Neutral

Text: "This movie was absolutely terrible."
Sentiment:""",
        
        "Chain-of-thought": f"""
{task}

Let's think step by step:
1. Identify key sentiment words
2. Determine if they are positive or negative
3. Give the final sentiment

Analysis:""",
        
        "System prompt": f"""
System: You are a sentiment analysis expert. Respond with only one word: Positive, Negative, or Neutral.

User: {task}

Assistant:"""
    }
    
    for name, prompt in strategies.items():
        print(f"=== {name} ===")
        print(prompt)
        print("\n" + "-" * 50 + "\n")

show_prompting_strategies()

---

## Step 5: How Engineers Make LLMs Useful

In real systems, you typically combine:

| Technique | Purpose | Example |
|-----------|---------|--------|
| **Prompting** | Instructions + examples + constraints | System prompts, few-shot examples |
| **Evaluation** | Tests for correctness, safety, style | Automated evals, human review |
| **RAG** | Add external knowledge at inference | Vector DB + retrieval + generation |
| **Fine-tuning** | Consistent behavior, formats, domain | LoRA on domain data |
| **Guardrails** | Safety, format enforcement | Output parsers, content filters |
| **Tool use** | Extend capabilities | Code execution, API calls, search |

In [ ]:
# Example: Simple output parsing / guardrail

import json
import re

def parse_json_response(response: str) -> dict:
    """
    Extract JSON from an LLM response.
    LLMs often wrap JSON in markdown code blocks.
    """
    # Try to find JSON in code blocks
    json_match = re.search(r'```(?:json)?\s*([\s\S]*?)```', response)
    if json_match:
        json_str = json_match.group(1).strip()
    else:
        # Try to parse the whole response
        json_str = response.strip()
    
    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        return {"error": f"Failed to parse JSON: {e}", "raw": response}

# Test with different response formats
test_responses = [
    '{"sentiment": "negative", "confidence": 0.95}',
    '```json\n{"sentiment": "negative", "confidence": 0.95}\n```',
    'The sentiment is negative. Here is the structured output:\n```json\n{"sentiment": "negative", "confidence": 0.95}\n```',
    'This is not valid JSON at all!',
]

print("Parsing different LLM response formats:\n")
for resp in test_responses:
    parsed = parse_json_response(resp)
    print(f"Input: {resp[:50]}...")
    print(f"Parsed: {parsed}\n")

In [ ]:
# Example: Simple RAG concept

def simple_rag_demo():
    """
    Demonstrate the RAG (Retrieval-Augmented Generation) concept.
    In practice, you'd use a vector database and embeddings.
    """
    
    # Our "knowledge base" (in reality, this would be a vector DB)
    documents = [
        {"id": 1, "content": "The company was founded in 2020 by Jane Smith."},
        {"id": 2, "content": "Our main product is an AI-powered writing assistant."},
        {"id": 3, "content": "The company is headquartered in San Francisco."},
        {"id": 4, "content": "We have 50 employees as of 2024."},
    ]
    
    def simple_retrieve(query, docs, top_k=2):
        """Simple keyword-based retrieval (real RAG uses embeddings)."""
        query_words = set(query.lower().split())
        scored = []
        for doc in docs:
            doc_words = set(doc["content"].lower().split())
            score = len(query_words & doc_words)
            scored.append((score, doc))
        scored.sort(reverse=True)
        return [doc for _, doc in scored[:top_k]]
    
    # User question
    question = "Who founded the company and where is it located?"
    
    # Retrieve relevant documents
    retrieved = simple_retrieve(question, documents)
    
    # Build the augmented prompt
    context = "\n".join([f"- {doc['content']}" for doc in retrieved])
    
    augmented_prompt = f"""
Answer the question based on the following context:

Context:
{context}

Question: {question}

Answer:"""
    
    print("=== RAG Demo ===")
    print(f"\nUser question: {question}")
    print(f"\nRetrieved documents:")
    for doc in retrieved:
        print(f"  - {doc['content']}")
    print(f"\nAugmented prompt sent to LLM:")
    print(augmented_prompt)

simple_rag_demo()

---

## Summary

### Key Takeaways

1. **LLMs predict the next token** — everything else builds on this
2. **Tokenization matters** — affects cost, context limits, and model behavior
3. **Training vs inference** — models learn patterns during training, apply them at inference
4. **Transformers use attention** — allows looking at all previous tokens in parallel
5. **Transfer learning** — start from pretrained models, adapt for your use case
6. **Engineering makes LLMs useful** — prompting, RAG, evaluation, guardrails

### Next Steps

- Experiment with tokenization on your own text
- Try different prompting strategies with a real LLM API
- Build a simple RAG system with a vector database
- Explore fine-tuning with LoRA on a small dataset

---

## Exercises

1. **Tokenization exploration**: Use `tiktoken` to compare how different languages tokenize. Why do some languages use more tokens per word?

2. **Bigram model extension**: Extend the `SimpleBigramLM` to a trigram model (look at 2 previous tokens). Does it generate better text?

3. **Attention visualization**: Modify the attention demo to use a causal mask (each token can only attend to previous tokens). How does this change the attention pattern?

4. **Prompt engineering**: Write prompts for the same task using zero-shot, few-shot, and chain-of-thought approaches. Compare the results with a real LLM.